In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
catalog_name = "ecommerce"

In [0]:
df = spark.table("ecommerce.silver.slv_order_items")
df.show(10)

In [0]:
df = df.withColumn("gross_amount", col("quantity") * col("unit_price"))

In [0]:
df = df.withColumn("discount_amount", col("gross_amount") * col("discount_pct")/100)
df.show(10)

In [0]:
df = df.withColumn("sale_amount", col("gross_amount") - col("discount_amount"))
df.show(10)

In [0]:
df = df.withColumn("date_id", date_format(col("dt"), "yyyyMMdd").cast("int"))
df.show(10)

In [0]:
df = df.withColumn("coupon_flag", when(col("coupon_code").isNotNull(), 1).otherwise(0))
df.show(10)

In [0]:
fx_rates = {
    "INR": 0.01044,
    "GBP": 1.3246,
    "AED": 0.2723,
    "CAD": 0.7071,
    "USD": 1.0,
    "AUD": 0.7026,
    "SGD": 0.7827
}

rates = [(currency, float(rate)) for currency, rate in fx_rates.items()]
rates_df = spark.createDataFrame(rates, ["currency", "USD_rate"])
rates_df.show()

In [0]:
df = df.join(rates_df, df["unit_price_currency"] == rates_df["currency"], "left")\
    .withColumn("sale_amount_USD", col("sale_amount")*col("USD_rate"))\
    .withColumn("sale_amount_USD", round(col("sale_amount_USD"), 2))

df.show(10)

In [0]:
orders_gold_df = df.select(
    col("date_id"),
    col("dt").alias("transaction_date"),
    col("order_ts").alias("transaction_ts"),
    col("order_id").alias("transaction_id"),
    col("customer_id"),
    col("item_seq").alias("seq_no"),
    col("product_id"),
    col("channel"),
    col("coupon_code"),
    col("coupon_flag"),
    col("unit_price_currency"),
    col("quantity"),
    col("unit_price"),
    col("gross_amount"),
    col("discount_pct").alias("discount_percent"),
    col("discount_amount"),
    col("tax_amount"),
    col("sale_amount").alias("net_amount"),
    col("sale_amount_USD").alias("net_amount_USD")
)

orders_gold_df.show(10)

In [0]:
orders_gold_df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.gold.gld_fact_order_items")